In [ ]:
# Cell 1: Setup and imports
from __future__ import annotations

import csv
import importlib.util
import math
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import ecole
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import torch_geometric
from scipy.stats import pearsonr, spearmanr

ROOT = Path.cwd().resolve()
WORK_DIR = ROOT / "vast_work"
ECOLE_REPO = WORK_DIR / "learn2branch-ecole"
ORIGINAL_REPO = WORK_DIR / "learn2branch"
DATA_DIR = ROOT / "data" / "instances"
RESULTS_DIR = ROOT / "results" / "auction_anchor"
MODEL_DIR = ROOT / "models" / "baseline_cauctions"
MODEL_PATH = MODEL_DIR / "train_params.pkl"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CLASS_DIRS = {
    "cauctions_valid": DATA_DIR / "eval_cauctions",
    "indset": DATA_DIR / "eval_indset",
    "setcover": DATA_DIR / "eval_setcover",
    "facilities": DATA_DIR / "eval_facilities",
}
REFERENCE_DIR = DATA_DIR / "reference_cauctions"

for path in [ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"root: {ROOT}")
print(f"model: {MODEL_PATH} exists={MODEL_PATH.exists()}")
print(f"ecole: {ecole.__version__}")
print(f"torch: {torch.__version__}")
print(f"torch_geometric: {torch_geometric.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


In [ ]:
# Cell 2: Prepare official L2B instances
# This cell creates a clean local instance set for the auction-anchor experiment.
# It reuses instances produced by runners/vast_cauctions_runner.py when available,
# and generates missing official L2B families from vast_work/learn2branch.

TARGET_PER_CLASS = 100
REFERENCE_SIZE = 100


def run_cmd(args, cwd=None):
    print("+", " ".join(map(str, args)))
    subprocess.run(args, cwd=cwd, check=True)


def ensure_repos():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    if not ORIGINAL_REPO.exists():
        run_cmd(["git", "clone", "--depth", "1", "https://github.com/ds4dm/learn2branch.git", str(ORIGINAL_REPO)])
        shutil.rmtree(ORIGINAL_REPO / ".git", ignore_errors=True)
    if not ECOLE_REPO.exists():
        run_cmd(["git", "clone", "--depth", "1", "https://github.com/ds4dm/learn2branch-ecole.git", str(ECOLE_REPO)])
        shutil.rmtree(ECOLE_REPO / ".git", ignore_errors=True)


def load_generator():
    generator_path = ORIGINAL_REPO / "01_generate_instances.py"
    if not generator_path.exists():
        raise FileNotFoundError(f"missing generator: {generator_path}")
    sys.path.insert(0, str(generator_path.parent))
    spec = importlib.util.spec_from_file_location("l2b_generator", generator_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def instance_files(path: Path) -> list[Path]:
    files = list(path.glob("*.lp")) + list(path.glob("*.mps"))
    return sorted(p for p in files if " " not in p.stem)


def clear_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    for file in instance_files(path):
        file.unlink()


def copy_first(source: Path, dest: Path, prefix: str, count: int) -> bool:
    files = instance_files(source)
    if len(files) < count:
        return False
    clear_dir(dest)
    for idx, src in enumerate(files[:count]):
        target = dest / f"{prefix}_{idx:03d}{src.suffix}"
        shutil.copy2(src, target)
    print(f"{dest.name}: copied {count} from {source}")
    return True


def generate_cauctions(module, dest: Path, count: int, seed: int, n_items=100, n_bids=500):
    clear_dir(dest)
    rng = np.random.RandomState(seed)
    for idx in range(count):
        module.generate_cauctions(rng, str(dest / f"cauctions_{idx:03d}.lp"), n_items=n_items, n_bids=n_bids, add_item_prob=0.7)
        if (idx + 1) % 25 == 0 or idx + 1 == count:
            print(f"{dest.name}: generated {idx + 1}/{count}")


def generate_setcover(module, dest: Path, count: int, seed: int):
    clear_dir(dest)
    rng = np.random.RandomState(seed)
    for idx in range(count):
        module.generate_setcover(500, 1000, 0.05, str(dest / f"setcover_{idx:03d}.lp"), rng, max_coef=100)
        if (idx + 1) % 25 == 0 or idx + 1 == count:
            print(f"{dest.name}: generated {idx + 1}/{count}")


def generate_indset(module, dest: Path, count: int, seed: int):
    clear_dir(dest)
    rng = np.random.RandomState(seed)
    for idx in range(count):
        graph = module.Graph.barabasi_albert(500, 4, rng)
        module.generate_indset(graph, str(dest / f"indset_{idx:03d}.lp"))
        if (idx + 1) % 25 == 0 or idx + 1 == count:
            print(f"{dest.name}: generated {idx + 1}/{count}")


def generate_facilities(module, dest: Path, count: int, seed: int):
    clear_dir(dest)
    rng = np.random.RandomState(seed)
    # The official generator function uses the global name rng internally.
    module.rng = rng
    for idx in range(count):
        module.generate_capacited_facility_location(rng, str(dest / f"facilities_{idx:03d}.lp"), n_customers=100, n_facilities=100, ratio=5)
        if (idx + 1) % 25 == 0 or idx + 1 == count:
            print(f"{dest.name}: generated {idx + 1}/{count}")


ensure_repos()
generator = load_generator()

# Reference and in-distribution eval use auction instances. Prefer the exact Vast runner folders if present.
reference_sources = [
    ECOLE_REPO / "data" / "instances" / "cauctions" / "train_100_500",
    ORIGINAL_REPO / "data" / "instances" / "cauctions" / "train_100_500",
]
if not any(copy_first(src, REFERENCE_DIR, "cauctions_reference", REFERENCE_SIZE) for src in reference_sources):
    generate_cauctions(generator, REFERENCE_DIR, REFERENCE_SIZE, seed=1000)

cauction_sources = [
    ECOLE_REPO / "data" / "instances" / "cauctions" / "valid_100_500",
    ECOLE_REPO / "data" / "instances" / "cauctions" / "test_100_500",
    ORIGINAL_REPO / "data" / "instances" / "cauctions" / "valid_100_500",
]
if not any(copy_first(src, EVAL_CLASS_DIRS["cauctions_valid"], "cauctions", TARGET_PER_CLASS) for src in cauction_sources):
    generate_cauctions(generator, EVAL_CLASS_DIRS["cauctions_valid"], TARGET_PER_CLASS, seed=1001)

# OOD official L2B classes.
setcover_sources = [ORIGINAL_REPO / "data" / "instances" / "setcover" / "transfer_500r_1000c_0.05d"]
if not any(copy_first(src, EVAL_CLASS_DIRS["setcover"], "setcover", TARGET_PER_CLASS) for src in setcover_sources):
    generate_setcover(generator, EVAL_CLASS_DIRS["setcover"], TARGET_PER_CLASS, seed=1002)

indset_sources = [ORIGINAL_REPO / "data" / "instances" / "indset" / "transfer_500_4"]
if not any(copy_first(src, EVAL_CLASS_DIRS["indset"], "indset", TARGET_PER_CLASS) for src in indset_sources):
    generate_indset(generator, EVAL_CLASS_DIRS["indset"], TARGET_PER_CLASS, seed=1003)

facility_sources = [ORIGINAL_REPO / "data" / "instances" / "facilities" / "transfer_100_100_5"]
if not any(copy_first(src, EVAL_CLASS_DIRS["facilities"], "facilities", TARGET_PER_CLASS) for src in facility_sources):
    generate_facilities(generator, EVAL_CLASS_DIRS["facilities"], TARGET_PER_CLASS, seed=1004)

inventory_rows = [{"role": "reference", "class": "cauctions", "folder": str(REFERENCE_DIR), "n_instances": len(instance_files(REFERENCE_DIR))}]
for class_name, folder in EVAL_CLASS_DIRS.items():
    inventory_rows.append({"role": "eval", "class": class_name, "folder": str(folder), "n_instances": len(instance_files(folder))})
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(RESULTS_DIR / "instance_inventory.csv", index=False)
print(inventory_df.to_string(index=False))

missing = inventory_df[
    ((inventory_df["role"] == "reference") & (inventory_df["n_instances"] < REFERENCE_SIZE))
    | ((inventory_df["role"] == "eval") & (inventory_df["n_instances"] < TARGET_PER_CLASS))
]
if not missing.empty:
    raise RuntimeError(f"Missing required instances:\n{missing.to_string(index=False)}")


In [ ]:
# Cell 3: Load auction-trained model
if not MODEL_PATH.exists():
    fallback = ECOLE_REPO / "model" / "cauctions" / "0" / "train_params.pkl"
    if fallback.exists():
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(fallback, MODEL_PATH)
        log_src = fallback.with_name("train_log.txt")
        if log_src.exists():
            shutil.copy2(log_src, MODEL_DIR / "train_log.txt")
    else:
        raise FileNotFoundError(f"Missing trained auction model weights: {MODEL_PATH}")

from models.learn2branch_gnn import GNNPolicy

device = "cuda:0" if torch.cuda.is_available() else "cpu"
policy = GNNPolicy().to(device)
policy.load_state_dict(torch.load(MODEL_PATH, map_location=device))
policy.eval()

print("loaded auction-trained GNNPolicy")
print(f"weights: {MODEL_PATH}")
print(f"device: {device}")


In [ ]:
# Cell 4: Build auction reference set and distance cache
from milp_distance.distance import extract_normalized_representation, greedy_instance_distance

REFERENCE_SEED = 0
REFERENCE_CACHE = RESULTS_DIR / "reference_reps.pkl"
DISTANCE_CACHE_CSV = RESULTS_DIR / "distance_cache.csv"

reference_candidates = instance_files(REFERENCE_DIR)
if len(reference_candidates) < REFERENCE_SIZE:
    raise RuntimeError(f"Need at least {REFERENCE_SIZE} auction reference instances in {REFERENCE_DIR}; found {len(reference_candidates)}")

rng_ref = np.random.RandomState(REFERENCE_SEED)
reference_paths = [reference_candidates[i] for i in rng_ref.choice(len(reference_candidates), size=REFERENCE_SIZE, replace=False)]
print(f"using {len(reference_paths)} auction reference instances from {REFERENCE_DIR}")

if REFERENCE_CACHE.exists():
    with REFERENCE_CACHE.open("rb") as handle:
        payload = pickle.load(handle)
    if payload.get("paths") == [str(p.resolve()) for p in reference_paths]:
        reference_reps = payload["reps"]
        print(f"loaded cached reference representations from {REFERENCE_CACHE}")
    else:
        reference_reps = None
else:
    reference_reps = None

if reference_reps is None:
    reference_reps = []
    for i, path in enumerate(reference_paths, start=1):
        print(f"reference {i}/{REFERENCE_SIZE}: {path.name}")
        reference_reps.append(extract_normalized_representation(path))
    with REFERENCE_CACHE.open("wb") as handle:
        pickle.dump({"paths": [str(p.resolve()) for p in reference_paths], "reps": reference_reps}, handle)

if DISTANCE_CACHE_CSV.exists():
    distance_cache_df = pd.read_csv(DISTANCE_CACHE_CSV)
    distance_cache = dict(zip(distance_cache_df["path"], distance_cache_df["distance"]))
else:
    distance_cache = {}


def save_distance_cache():
    pd.DataFrame(
        [{"path": path, "distance": distance} for path, distance in sorted(distance_cache.items())]
    ).to_csv(DISTANCE_CACHE_CSV, index=False)


def distribution_distance(mps_path) -> float:
    path = str(Path(mps_path).resolve())
    if path in distance_cache:
        return float(distance_cache[path])
    rep = extract_normalized_representation(path)
    distances = [greedy_instance_distance(rep, ref) for ref in reference_reps if ref is not None]
    finite = [distance for distance in distances if np.isfinite(distance)]
    distance = float(np.mean(finite)) if finite else float("inf")
    distance_cache[path] = distance
    if len(distance_cache) % 10 == 0:
        save_distance_cache()
    return distance

print(f"reference reps ready: {len(reference_reps)}")
print(f"distance cache entries: {len(distance_cache)}")


In [ ]:
# Cell 5: Branching evaluation function
SCIP_PARAMS = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 300.0,
    "limits/nodes": 10000,
    "timing/clocktype": 1,
}
FORCE_RELPSCOST_BASELINE = True


def tensorize_node_observation(obs, device):
    return (
        torch.from_numpy(obs.row_features.astype(np.float32)).to(device),
        torch.from_numpy(obs.edge_features.indices.astype(np.int64)).to(device),
        torch.from_numpy(obs.edge_features.values.astype(np.float32)).view(-1, 1).to(device),
        torch.from_numpy(obs.variable_features.astype(np.float32)).to(device),
    )


def solve_vanilla_scip(instance_path, time_limit=300):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    if FORCE_RELPSCOST_BASELINE:
        params["branching/relpscost/priority"] = 9999999
    env = ecole.environment.Configuring(scip_params=params)
    env.seed(0)
    wall_start = time.perf_counter()
    env.reset(str(instance_path))
    _, _, _, _, _ = env.step({})
    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "scip_nodes": int(model.getNNodes()),
        "scip_lps": int(model.getNLPs()),
        "scip_time": float(model.getSolvingTime()),
        "scip_walltime": float(walltime),
        "scip_status": str(model.getStatus()),
        "scip_gap": float(model.getGap()),
    }


def solve_ml_branching(instance_path, policy, device, time_limit=300):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    policy.eval()
    env = ecole.environment.Branching(
        observation_function=ecole.observation.NodeBipartite(),
        scip_params=params,
        pseudo_candidates=False,
    )
    env.seed(0)
    torch.manual_seed(0)
    wall_start = time.perf_counter()

    obs, action_set, _, done, _ = env.reset(str(instance_path))
    while not done:
        if action_set is None or len(action_set) == 0:
            raise RuntimeError("empty action_set before solve completion")
        with torch.no_grad():
            tensors = tensorize_node_observation(obs, device)
            logits = policy(*tensors)
            action_tensor_index = torch.as_tensor(action_set.astype(np.int64), device=device)
            selected = int(torch.argmax(logits[action_tensor_index]).item())
            action = int(action_set[selected])
        obs, action_set, _, done, _ = env.step(action)

    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "ml_nodes": int(model.getNNodes()),
        "ml_lps": int(model.getNLPs()),
        "ml_time": float(model.getSolvingTime()),
        "ml_walltime": float(walltime),
        "ml_status": str(model.getStatus()),
        "ml_gap": float(model.getGap()),
    }


def evaluate_instance(mps_path, policy, device, time_limit=300) -> dict | None:
    try:
        path = Path(mps_path)
        scip = solve_vanilla_scip(path, time_limit=time_limit)
        ml = solve_ml_branching(path, policy, device, time_limit=time_limit)
        scip_nodes = max(1, int(scip["scip_nodes"]))
        ml_nodes = int(ml["ml_nodes"])
        return {
            **scip,
            **ml,
            "degradation": float(ml_nodes / scip_nodes),
            "relative_node_count": float(ml_nodes / scip_nodes),
            "excess_nodes": int(ml_nodes - scip_nodes),
            "smoothed_rnc_100": float((ml_nodes + 100) / (scip_nodes + 100)),
        }
    except Exception as exc:
        print(f"FAILED {mps_path}: {type(exc).__name__}: {exc}")
        return None


In [ ]:
# Cell 6: Run auction-anchor experiment
RAW_RESULTS_CSV = RESULTS_DIR / "raw_results.csv"
FAILURES_CSV = RESULTS_DIR / "failures.csv"
N_PER_CLASS = 100
TIME_LIMIT = 300
OFFICIAL_CLASSES = [
    "cauctions_valid",
    "indset",
    "setcover",
    "facilities",
]


def first_n(paths, n=N_PER_CLASS):
    return [Path(path) for path in sorted(paths)[:n]]


test_sets = {
    class_name: first_n(instance_files(EVAL_CLASS_DIRS[class_name]), N_PER_CLASS)
    for class_name in OFFICIAL_CLASSES
}
test_sets = {name: paths for name, paths in test_sets.items() if paths}
for name, paths in test_sets.items():
    print(f"{name}: {len(paths)} instances")

if RAW_RESULTS_CSV.exists():
    results_df = pd.read_csv(RAW_RESULTS_CSV)
    completed = set(results_df["path"].astype(str))
    rows = results_df.to_dict("records")
    print(f"loaded {len(rows)} existing results from {RAW_RESULTS_CSV}")
else:
    completed = set()
    rows = []

if FAILURES_CSV.exists():
    failures_df = pd.read_csv(FAILURES_CSV)
    failed_paths = set(failures_df["path"].astype(str))
    failure_rows = failures_df.to_dict("records")
else:
    failed_paths = set()
    failure_rows = []

for class_name, paths in test_sets.items():
    for idx, path in enumerate(paths, start=1):
        resolved = str(path.resolve())
        if resolved in completed or resolved in failed_paths:
            continue
        try:
            distance = distribution_distance(path)
            metrics = evaluate_instance(path, policy, device, time_limit=TIME_LIMIT)
            if metrics is None:
                failure_rows.append({"class": class_name, "path": resolved, "reason": "evaluate_instance returned None"})
                pd.DataFrame(failure_rows).to_csv(FAILURES_CSV, index=False)
                failed_paths.add(resolved)
                continue
            row = {"class": class_name, "path": resolved, "distance": distance, **metrics}
            rows.append(row)
            completed.add(resolved)
            pd.DataFrame(rows).to_csv(RAW_RESULTS_CSV, index=False)
            save_distance_cache()
            print(f"{class_name} {idx}/{len(paths)} distance={distance:.4f} rnc={metrics['degradation']:.3f} scip={metrics['scip_nodes']} ml={metrics['ml_nodes']}")
        except Exception as exc:
            print(f"FAILED {class_name} {path.name}: {type(exc).__name__}: {exc}")
            failure_rows.append({"class": class_name, "path": resolved, "reason": f"{type(exc).__name__}: {exc}"})
            pd.DataFrame(failure_rows).to_csv(FAILURES_CSV, index=False)
            failed_paths.add(resolved)

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_CSV, index=False)
save_distance_cache()
print(f"saved {len(results_df)} rows to {RAW_RESULTS_CSV}")
if failure_rows:
    print(f"saved {len(failure_rows)} failures to {FAILURES_CSV}")


In [ ]:
# Cell 7: Analysis and plots
RAW_RESULTS_CSV = RESULTS_DIR / "raw_results.csv"
if not RAW_RESULTS_CSV.exists():
    raise RuntimeError("No raw results found. Run Cell 6 first.")

results_df = pd.read_csv(RAW_RESULTS_CSV)
results_df = results_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["distance", "degradation"])
results_df = results_df[results_df["degradation"] > 0].copy()
if results_df.empty:
    raise RuntimeError("No usable positive-degradation results found. Run Cell 6 first.")

results_df["log_degradation"] = np.log10(results_df["degradation"])
results_df["ml_worse"] = results_df["degradation"] > 1.0
results_df["scip_limited"] = results_df["scip_status"].astype(str).str.contains("limit", case=False, na=False)
results_df["ml_limited"] = results_df["ml_status"].astype(str).str.contains("limit", case=False, na=False)
classes = sorted(results_df["class"].unique())


def safe_corr(x, y, method="pearson"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or np.nanstd(x[mask]) == 0 or np.nanstd(y[mask]) == 0:
        return np.nan, np.nan
    if method == "spearman":
        return spearmanr(x[mask], y[mask])
    return pearsonr(x[mask], y[mask])


r_raw, p_raw = safe_corr(results_df["distance"], results_df["degradation"])
r_log, p_log = safe_corr(results_df["distance"], results_df["log_degradation"])
r_spear, p_spear = safe_corr(results_df["distance"], results_df["degradation"], method="spearman")
r_smooth, p_smooth = safe_corr(results_df["distance"], results_df["smoothed_rnc_100"])
r_excess, p_excess = safe_corr(results_df["distance"], results_df["excess_nodes"])

correlation_summary = pd.DataFrame([
    {"metric": "pearson_distance_rnc", "r": r_raw, "p_value": p_raw},
    {"metric": "pearson_distance_log10_rnc", "r": r_log, "p_value": p_log},
    {"metric": "spearman_distance_rnc", "r": r_spear, "p_value": p_spear},
    {"metric": "pearson_distance_smoothed_rnc_100", "r": r_smooth, "p_value": p_smooth},
    {"metric": "pearson_distance_excess_nodes", "r": r_excess, "p_value": p_excess},
])
correlation_summary.to_csv(RESULTS_DIR / "correlation_summary.csv", index=False)
print(correlation_summary.to_string(index=False))

summary_by_class = results_df.groupby("class").agg(
    n=("class", "size"),
    median_distance=("distance", "median"),
    mean_distance=("distance", "mean"),
    median_scip_nodes=("scip_nodes", "median"),
    median_ml_nodes=("ml_nodes", "median"),
    median_rnc=("degradation", "median"),
    mean_rnc=("degradation", "mean"),
    median_smoothed_rnc_100=("smoothed_rnc_100", "median"),
    median_excess_nodes=("excess_nodes", "median"),
    pct_ml_worse=("ml_worse", "mean"),
    pct_scip_limited=("scip_limited", "mean"),
    pct_ml_limited=("ml_limited", "mean"),
).reset_index().sort_values("median_distance")
summary_by_class.to_csv(RESULTS_DIR / "summary_by_class.csv", index=False)
print(summary_by_class.to_string(index=False))

# Scatter: distance vs log relative node count.
plt.figure(figsize=(10, 6))
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["distance"], subset["degradation"], label=class_name, alpha=0.72)
plt.yscale("log")
plt.axhline(1.0, color="red", linestyle="--", linewidth=1)
if results_df["distance"].nunique() > 1:
    coef = np.polyfit(results_df["distance"], results_df["log_degradation"], 1)
    xs = np.linspace(results_df["distance"].min(), results_df["distance"].max(), 100)
    ys = 10 ** (coef[0] * xs + coef[1])
    plt.plot(xs, ys, color="black", linewidth=2, label="linear fit in log space")
plt.xlabel("Mean Maudet distance to auction reference")
plt.ylabel("Relative node count: ML nodes / SCIP nodes")
plt.title("Auction-Anchor Distance vs ML Branching Transfer")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "scatter_distance_rnc_log.png", dpi=200)
plt.show()

# Class-level median plot.
plt.figure(figsize=(8, 5))
plt.scatter(summary_by_class["median_distance"], summary_by_class["median_rnc"], s=90)
for _, row in summary_by_class.iterrows():
    plt.annotate(row["class"], (row["median_distance"], row["median_rnc"]), xytext=(5, 5), textcoords="offset points")
plt.yscale("log")
plt.axhline(1.0, color="red", linestyle="--", linewidth=1)
plt.xlabel("Median Maudet distance to auction reference")
plt.ylabel("Median relative node count")
plt.title("Class Median Transfer Degradation")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_median_distance_vs_rnc.png", dpi=200)
plt.show()

# Boxplot by class ordered by distance.
ordered_classes = list(summary_by_class["class"])
plt.figure(figsize=(10, 5))
plt.boxplot([results_df[results_df["class"] == cls]["degradation"] for cls in ordered_classes], labels=ordered_classes, showfliers=False)
plt.yscale("log")
plt.axhline(1.0, color="red", linestyle="--", linewidth=1)
plt.ylabel("Relative node count")
plt.title("Relative Node Count by Class")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "box_rnc_by_class.png", dpi=200)
plt.show()

# Absolute node comparison.
plt.figure(figsize=(7, 7))
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["scip_nodes"], subset["ml_nodes"], label=class_name, alpha=0.72)
max_nodes = max(results_df["scip_nodes"].max(), results_df["ml_nodes"].max())
plt.plot([1, max_nodes], [1, max_nodes], color="black", linewidth=1)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("SCIP nodes")
plt.ylabel("ML branching nodes")
plt.title("SCIP vs Auction-Trained ML Branching Nodes")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "scip_vs_ml_nodes.png", dpi=200)
plt.show()

print(f"saved plots and summaries under {RESULTS_DIR}")
